# RSNA Knee Abnormality Detection — Colab pipeline

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vahid-karami/rsna-knee-mri/blob/main/notebooks/rsna_knee_colab.ipynb)

Detect 12 knee abnormalities (ACL/MCL tears, meniscal tears, osteoarthritis, effusion, synovitis, Baker's cyst, contusion, fracture) on MRI.

Only a few dozen training studies have expert labels, so the pipeline **learns from the radiology reports**:

1. **Report labelling** — a rule-based multilingual extractor + an open LLM (Qwen2.5) turn ~4,400 free-text reports into soft labels, checked against the expert labels.
2. **Image model** — a 2.5D multi-view CNN (sagittal / coronal / axial) trained on those soft labels.
3. **Honest evaluation** — macro ROC-AUC (the competition metric) on the expert-labelled studies, which are never used for training or model selection.

**Before you start:** `Runtime → Change runtime type → T4 GPU`, and accept the competition rules at
https://www.kaggle.com/competitions/rsna-knee-abnormality-detection/rules (the data download fails otherwise).

## 1. Setup

In [ ]:
REPO = "https://github.com/vahid-karami/rsna-knee-mri.git"   # <- your repo
!git clone -q $REPO /content/rsna-knee-mri 2>/dev/null || git -C /content/rsna-knee-mri pull -q
%cd /content/rsna-knee-mri
!pip install -q -r requirements.txt

Outputs go to Google Drive, so a disconnected Colab session loses nothing (the LLM labelling step resumes where it stopped).

In [ ]:
import os, sys
from google.colab import drive
drive.mount("/content/drive")

os.environ["KNEE_DATA_ROOT"] = "/content/data"
os.environ["KNEE_WORK_DIR"]  = "/content/drive/MyDrive/rsna-knee/work"
os.environ["KNEE_CACHE_DIR"] = "/content/cache"          # local disk is faster than Drive for training
os.makedirs(os.environ["KNEE_WORK_DIR"], exist_ok=True)
sys.path.insert(0, "/content/rsna-knee-mri/src")

### Kaggle API key
Kaggle → Settings → API → *Create New Token*. Then either add `KAGGLE_USERNAME` and `KAGGLE_KEY` as Colab **secrets** (key icon on the left), or upload `kaggle.json` when the cell asks.

In [ ]:
import pathlib
try:
    from google.colab import userdata
    os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
    os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")
except Exception:
    from google.colab import files
    up = files.upload()  # choose kaggle.json
    cfg = pathlib.Path.home() / ".kaggle"; cfg.mkdir(exist_ok=True)
    (cfg / "kaggle.json").write_bytes(up["kaggle.json"]); (cfg / "kaggle.json").chmod(0o600)

## 2. Download the tables (reports, labels, series metadata: a few MB, not the 250 GB of images)

In [ ]:
%%bash
mkdir -p /content/data && cd /content/data
for f in train.csv train_series.csv test.csv test_series.csv sample_submission.csv; do
  kaggle competitions download -c rsna-knee-abnormality-detection -f $f -p . -q || echo "could not get $f"
done
for z in *.zip; do [ -e "$z" ] && unzip -oq "$z" && rm "$z"; done
ls -lh

## 3. Explore the data

In [ ]:
import json
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from knee import config as C
from knee.data import load_train, load_series, gold_mask

labels = C.load_label_names()
train = load_train(); series = load_series("train")
gold = gold_mask(train, labels)
print(f"{len(train)} studies | {len(series)} series | {gold.sum()} expert-labelled studies")
print(series[C.PLANE_COL].value_counts())
train[gold][labels].mean().sort_values().plot.barh(title="Prevalence in expert-labelled studies", figsize=(6, 4));

In [ ]:
print(train[C.REPORT_COL].str.len().describe())
for r in train[C.REPORT_COL].sample(2, random_state=0):
    print("-" * 80); print(r[:1200])

## 4. Labels from reports
### 4a. Rule-based baseline (seconds)

In [ ]:
!python -m knee.build_labels

### 4b. LLM labeller (Qwen2.5-3B-Instruct on the T4)
Try 32 reports first to check speed and output, then run on all of them (about 1-1.5 h on a T4; if Colab disconnects, re-run the cell and it resumes).
For better labels with more time, use `--model Qwen/Qwen2.5-7B-Instruct --batch-size 8`.

In [ ]:
!python -m knee.report_llm --limit 32 --batch-size 16

In [ ]:
from itertools import islice
with open(f"{os.environ['KNEE_WORK_DIR']}/labels/llm_answers.jsonl") as f:
    for line in islice(f, 2):
        print(json.loads(line)["answer"][:500], "\n")

In [ ]:
!python -m knee.report_llm --batch-size 16

In [ ]:
!python -m knee.build_labels   # compares rules vs LLM vs both against the experts and keeps the best

## 5. MRI images
The full image set (~250 GB) does not fit on Colab, so it is preprocessed **once on Kaggle** (where the data is mounted) into a ~4 GB cache of 1,500 studies.

1. On Kaggle: *Create → New Notebook* → *File → Import Notebook* → `notebooks/kaggle_preprocess.ipynb` from this repo. *Add Input* → the competition. Turn **Internet on**. Name the notebook `rsna-knee-preprocess`, then *Save Version → Save & Run All*.
2. When it finishes, run the next cell here to download its output.

In [ ]:
%%bash
KAGGLE_NOTEBOOK="$KAGGLE_USERNAME/rsna-knee-preprocess"     # <- change if you named it differently
mkdir -p /content/cache_dl /content/cache
kaggle kernels output "$KAGGLE_NOTEBOOK" -p /content/cache_dl
tar -xf /content/cache_dl/knee_cache.tar -C /content/cache && rm -rf /content/cache_dl
ls /content/cache | wc -l

In [ ]:
import glob
d = np.load(sorted(glob.glob("/content/cache/*.npz"))[0]); vol = d["vol"]
fig, ax = plt.subplots(1, 3, figsize=(12, 4))
for v, name in enumerate(C.VIEWS):
    ax[v].imshow(vol[v, vol.shape[1] // 2], cmap="gray"); ax[v].set_title(name); ax[v].axis("off")

## 6. Train the image model (about 5-8 min per epoch on a T4 for 1,500 studies)
If the GPU runs out of memory, add `--grad-checkpointing` or use `--slices 12`.

In [ ]:
!python -m knee.train_image --epochs 10 --batch-size 4 --accum 2 --slices 16 --backbone efficientnet_b0

## 7. Results

In [ ]:
res = json.load(open(f"{os.environ['KNEE_WORK_DIR']}/models/results.json"))
h = pd.DataFrame(res["history"])
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].plot(h.epoch, h.val_auc, "o-", label="held-out (report labels)")
ax[0].plot(h.epoch, h.gold_auc, "s-", label="expert labels")
ax[0].set(xlabel="epoch", ylabel="macro ROC-AUC", title="Validation"); ax[0].legend(); ax[0].grid(alpha=.3)
pl = pd.Series(res["gold_per_label_auc"]).dropna().sort_values()
ax[1].barh(pl.index, pl.values, color="#3b6ea5"); ax[1].axvline(.5, ls="--", c="gray")
ax[1].set(xlim=(0, 1), title=f"Per-finding AUC on expert labels (macro {res['gold_macro_auc']:.3f})")
plt.tight_layout(); plt.savefig(f"{os.environ['KNEE_WORK_DIR']}/results.png", dpi=150)
print({k: res[k] for k in ("best_val_auc", "gold_macro_auc", "n_train", "n_gold")})

In [ ]:
pd.read_csv(f"{os.environ['KNEE_WORK_DIR']}/labels/label_agreement.csv", index_col=0).round(3)

## 8. Next steps
* Train several models with different seeds or backbones (`--backbone convnext_nano`, `tf_efficientnetv2_s`) and average them.
* Preprocess all ~4,400 studies (`MAX_STUDIES = 0`) on Kaggle.
* **Late submission:** upload `best.pt` as a Kaggle dataset, attach it and this repo to a Kaggle notebook (internet off), and run `python -m knee.predict --checkpoints /kaggle/input/<dataset>/*.pt`.